In [1]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM , AutoTokenizer , GenerationConfig , TrainingArguments , Trainer
import torch
import time
import evaluate
import pandas as pd
import numpy as np
from dotenv import load_dotenv

W1020 13:52:42.304000 17644 venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
load_dotenv()

True

In [3]:
huggingface_dataset_name="knkarthick/dialogsum"

dataset=load_dataset(huggingface_dataset_name)

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [5]:
dataset.save_to_disk("./dataset")

Saving the dataset (0/1 shards):   0%|          | 0/12460 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1500 [00:00<?, ? examples/s]

## Loading original/base model

In [6]:
model_name="google/flan-t5-base"

original_model=AutoModelForSeq2SeqLM.from_pretrained(pretrained_model_name_or_path=model_name,dtype=torch.bfloat16)

tokenizer=AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [7]:
print("Model dtype:", next(original_model.parameters()).dtype)
print("CUDA available:", torch.cuda.is_available())

Model dtype: torch.bfloat16
CUDA available: True


In [8]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params=0
    all_model_params=0
    for _,param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"Trainable model parameters : {trainable_model_params}\nAll model parameters: {all_model_params}\nPercentage of trainable model parameters {(trainable_model_params / all_model_params)*100}"

In [9]:
print(print_number_of_trainable_model_parameters(original_model))

Trainable model parameters : 247577856
All model parameters: 247577856
Percentage of trainable model parameters 100.0


In [10]:
index=200

dialogue=dataset['test'][index]['dialogue']
summary=dataset['test'][index]['summary']

prompt= f"""
        Summarize the following conversation
        {dialogue}

        Summary:
        """
inputs=tokenizer(prompt,return_tensors="pt")
output=tokenizer.decode(
    original_model.generate(
        inputs['input_ids'],
        max_new_tokens=200
    )[0],
    skip_special_tokens=True
)

dash_line="-".join("" for i in range(100))

print(dash_line)
print(f"Input prompt:\n {prompt}")
print(dash_line)
print(f"Baseline Human summary:\n{summary}")
print(dash_line)
print(f"Model generation - ZERO SHOT:\n {output}")

---------------------------------------------------------------------------------------------------
Input prompt:
 
        Summarize the following conversation
        #Person1#: Have you considered upgrading your system?
#Person2#: Yes, but I'm not sure what exactly I would need.
#Person1#: You could consider adding a painting program to your software. It would allow you to make up your own flyers and banners for advertising.
#Person2#: That would be a definite bonus.
#Person1#: You might also want to upgrade your hardware because it is pretty outdated now.
#Person2#: How can we do that?
#Person1#: You'd probably need a faster processor, to begin with. And you also need a more powerful hard disc, more memory and a faster modem. Do you have a CD-ROM drive?
#Person2#: No.
#Person1#: Then you might want to add a CD-ROM drive too, because most new software programs are coming out on Cds.
#Person2#: That sounds great. Thanks.

        Summary:
        
------------------------------------

In [11]:
def tokenize_function(example):
    start_prompt="Summarize the following conversation.\n\n"
    end_prompt="\n\nSummary"
    prompt=[start_prompt+dialogue+end_prompt for dialogue in example["dialogue"]]
    example["input_ids"]=tokenizer(prompt, padding="max_length",truncation=True,return_tensors="pt").input_ids
    example["labels"]=tokenizer(example["summary"],padding="max_length",truncation=True,return_tensors="pt").input_ids
    return example

In [12]:
tokenized_datasets=dataset.map(tokenize_function,batched=True)
tokenized_datasets=tokenized_datasets.remove_columns(["id","topic","dialogue","summary"])

In [13]:
print(f"Shapes of the datasets:")
print(f"Training: {tokenized_datasets['train'].shape}")
print(f"Validation: {tokenized_datasets['validation'].shape}")
print(f"Test: {tokenized_datasets['test'].shape}")

Shapes of the datasets:
Training: (12460, 2)
Validation: (500, 2)
Test: (1500, 2)


In [14]:
tokenized_datasets=tokenized_datasets.filter(lambda example,index: index % 100 == 0,with_indices=True)

In [15]:
print(f"Shapes of the datasets:")
print(f"Training: {tokenized_datasets['train'].shape}")
print(f"Validation: {tokenized_datasets['validation'].shape}")
print(f"Test: {tokenized_datasets['test'].shape}")

Shapes of the datasets:
Training: (125, 2)
Validation: (5, 2)
Test: (15, 2)


In [16]:
print(tokenized_datasets)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 125
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 5
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 15
    })
})


## Fine tuning base model on dialogsum dataset and only on 1 percent of dataset

In [17]:
output_dir=f"../data/dialogue-summary-training-{str(int(time.time()))}"

training_args=TrainingArguments(
    output_dir=output_dir,
    learning_rate=1e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    use_cpu=False,
    per_device_train_batch_size=1,
    bf16=True,
    fp16=False
)

trainer=Trainer(
    model=original_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"]
)

In [18]:
trainer.train()

Step,Training Loss
100,48.001304
200,46.459995
300,45.760752


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=375, training_loss=46.44151236979167, metrics={'train_runtime': 80.3941, 'train_samples_per_second': 4.665, 'train_steps_per_second': 4.665, 'total_flos': 256784007168000.0, 'train_loss': 46.44151236979167, 'epoch': 3.0})

In [19]:
trainer.save_model("../models/flan-t5-dialogsum")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [20]:
instruct_model=AutoModelForSeq2SeqLM.from_pretrained("../models/flan-t5-dialogsum").to("cuda")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [21]:
print("Original model:", next(original_model.parameters()).device)
print("Fine-tuned model:", next(instruct_model.parameters()).device)

Original model: cuda:0
Fine-tuned model: cuda:0


In [22]:
dialogue=dataset['test'][index]['dialogue']
human_baseline_summary=dataset['test'][index]['summary']

prompt=f"""
        Summarize the following conversation
        {dialogue}
        Summary:
"""
input_ids=tokenizer(prompt,return_tensors="pt").input_ids
input_ids=input_ids.to(original_model.device)
original_model_outputs=original_model.generate(input_ids=input_ids,generation_config=GenerationConfig(max_new_tokens=200,num_beams=1))
original_model_text_output=tokenizer.decode(original_model_outputs[0],skip_special_tokens=True)

instruct_model_outputs=instruct_model.generate(input_ids=input_ids,generation_config=GenerationConfig(max_new_tokens=200,num_beams=1))

instruct_model_text_output=tokenizer.decode(instruct_model_outputs[0],skip_special_tokens=True)

## Comparing base model and fine-tuned model

In [23]:
print(dash_line)
print(f"HUMAN BASELINE SUMMARY \n{human_baseline_summary}")
print(dash_line)
print(f"ORIGINAL MODEL \n{original_model_text_output}")
print(dash_line)
print(f"FINE TUNED MODEL\n{instruct_model_text_output}")

---------------------------------------------------------------------------------------------------
HUMAN BASELINE SUMMARY 
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.
---------------------------------------------------------------------------------------------------
ORIGINAL MODEL 
#Person2#: Yes, I'm thinking of upgrading my system.
---------------------------------------------------------------------------------------------------
FINE TUNED MODEL
#Person1#: I'm thinking of upgrading my computer.
